# rank0-only-side-effects — faded example 1: Guard checkpoint write with if rank == 0

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `rank0-only-side-effects`. The last cell reports your progress on the `Distributed: rank-0-only side effects` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: rank-0-only side effects` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`rank0-only-side-effects`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rank0-only-side-effects"
DD_SUBTOPIC = "Distributed: rank-0-only side effects"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In distributed training, checkpoint writes must be guarded so only rank 0 writes to the shared file system. The guard is exactly `if rank == 0:` — not `if rank % world_size == 0:` or any other variant. Code outside this guard runs on every rank (per-rank computation); code inside runs exactly once.

## Faded exercise 1

### Exercise — Guard checkpoint write with if rank == 0

Complete `epoch_end(rank, world_size, model_state, ckpt_fn, log_fn, per_rank_fn)`. The per-rank callback runs unconditionally. The checkpoint and log callbacks run only on rank 0.

Fill in the rank-0 guard.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
def epoch_end(rank, world_size, model_state, loss, ckpt_fn, log_fn, per_rank_fn):
    per_rank_fn(rank, loss)
    if None:  # TODO: fill in this step — read the prompt cell above
        ckpt_fn(model_state)
        log_fn(f'loss={loss:.4f}')

# Test
ckpts = []
logs = []
per = []
for r in range(4):
    epoch_end(r, 4, {}, 0.3, ckpts.append, logs.append, lambda rk, l: per.append(rk))
print(len(ckpts), len(logs), len(per))


def _test():
    ckpts = []
    logs = []
    per = []
    for r in range(6):
        epoch_end(r, 6, {'w': 1.0}, 0.25,
                  ckpts.append, logs.append,
                  lambda rk, l: per.append(rk))
    assert len(ckpts) == 1, f'expected 1 checkpoint, got {len(ckpts)}'
    assert len(logs) == 1, f'expected 1 log, got {len(logs)}'
    assert len(per) == 6, f'expected 6 per-rank calls, got {len(per)}'
    assert set(per) == set(range(6)), 'every rank must call per_rank_fn'


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def epoch_end(rank, world_size, model_state, loss, ckpt_fn, log_fn, per_rank_fn):
    per_rank_fn(rank, loss)
    if rank == 0:
        ckpt_fn(model_state)
        log_fn(f'loss={loss:.4f}')

# Test
ckpts = []
logs = []
per = []
for r in range(4):
    epoch_end(r, 4, {}, 0.3, ckpts.append, logs.append, lambda rk, l: per.append(rk))
print(len(ckpts), len(logs), len(per))
```
</details>